# LEXICON-X II | Notebook de trabajo
By Erasmo Gomez Montoya
Sube y extrae el ZIP; usa CPU para baseline.

In [1]:
# En Colab: subir y extraer el ZIP de estudiante en /content; cambiar directorio si corresponde.
# Los archivos CSV deben estar junto al notebook.
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from pathlib import Path
D=pd.read_csv('corpus_documentos.csv'); V=pd.read_csv('validation_qa.csv',dtype={'answer':str}).fillna('')
X=D.text.tolist(); vectorizer=TfidfVectorizer(ngram_range=(1,2)); M=vectorizer.fit_transform(X)
def lexical(q,k=5):
    scores=linear_kernel(vectorizer.transform([q]),M).ravel()
    return np.argsort(-scores,kind='stable')[:k]
def evaluate(pred,gold):
    assert pred.qid.is_unique and set(pred.qid)==set(gold.qid), 'IDs incompletos o duplicados'
    z=gold.merge(pred,on='qid',suffixes=('_gold','_pred'))
    exact=(z.answer_gold.astype(str).str.strip().str.lower()==z.answer_pred.astype(str).str.strip().str.lower())
    def hit(r):
        g=set(str(r.evidence_ids).split('|'))-{'','nan'}; p=set(str(r.citations).split('|'))-{'','nan'}
        return bool(g&p) if g else not p
    h=z.apply(hit,axis=1)
    return {'EM':float(exact.mean()),'evidence_hit':float(h.mean()),'joint':float((exact&h).mean()),'by_kind':exact.groupby(z.kind).mean().to_dict()}


In [2]:
display(D.sample(8,random_state=42)); display(V.head()); print(D.groupby(["language","source"]).size())

,doc_id,site,topic,language,year,source,reliability,text
1116,D186-0,ER-186,energía,es,2048,oficial,0.95,El registro oficial de ER-186 confirma 28 celd...
1368,D228-0,ER-228,archivo,es,2048,oficial,0.95,El registro oficial de ER-228 confirma 42 tabl...
422,D070-2,ER-070,agricultura,es,2047,oficial,0.90,El registro anterior de ER-070 indica 100 semi...
413,D068-5,ER-068,archivo,es,2048,archivo,0.70,El archivo de ER-068 registra actividades de a...
451,D075-1,ER-075,agricultura,er,2048,oficial,0.95,ER-075 sava 38 ena.
861,D143-3,ER-143,archivo,er,2047,oficial,0.90,ER-143 kiri 87 ena.
1063,D177-1,ER-177,transporte,er,2048,oficial,0.95,ER-177 nora 42 ena.
741,D123-3,ER-123,archivo,er,2047,oficial,0.90,ER-123 kiri 84 ena.


,qid,question,answer,evidence_ids,kind
0,Q148-3,¿Quién dirige ER-148?,SIN_EVIDENCIA,,sin_evidencia
1,Q128-3,¿Quién dirige ER-128?,SIN_EVIDENCIA,,sin_evidencia
2,Q132-2,¿Qué cantidad de vehículos registraba ER-132 e...,95,D132-2|D132-3,historico
3,Q164-2,¿Qué cantidad de depósitos registraba ER-164 e...,38,D164-2|D164-3,historico
4,Q157-0,¿Cuántas vehículos hay disponibles actualmente...,25,D157-0|D157-1,actual


language  source 
er        oficial    500
es        archivo    250
          oficial    500
          rumor      250
dtype: int64


## Hipótesis
Escribe hipótesis, variable independiente, métricas y criterio de éxito.

In [3]:
print(D.iloc[lexical(V.question.iloc[0])][["doc_id","text"]])

     doc_id                                               text
889  D148-1                                ER-148 kiri 49 ena.
891  D148-3                                ER-148 kiri 56 ena.
893  D148-5  El archivo de ER-148 registra actividades de a...
890  D148-2  El registro anterior de ER-148 indica 56 tabli...
888  D148-0  El registro oficial de ER-148 confirma 49 tabl...


## Baseline y alternativa
Implementa predict y run. La salida debe tener qid, answer, citations (IDs separados por |). Usa SIN_EVIDENCIA sin citas cuando no exista respuesta. Compara ambos enfoques con evaluate.

In [4]:
def predict(question):
    raise NotImplementedError("Implementar respuesta y citas")

## Recursos opcionales: embeddings y generación
Descomenta instalación si es necesaria. No son obligatorios; mide latencia y calidad.

In [ ]:
# Opcional: ejecutar en Colab con Internet y GPU si está disponible.
# %pip install -q sentence-transformers transformers sentencepiece
from sentence_transformers import SentenceTransformer
encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
E=encoder.encode(X,normalize_embeddings=True,show_progress_bar=True)
def dense(q,k=5):
    v=encoder.encode([q],normalize_embeddings=True)[0]
    return np.argsort(-(E@v))[:k]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
name='google/flan-t5-small'; tok=AutoTokenizer.from_pretrained(name); model=AutoModelForSeq2SeqLM.from_pretrained(name)
device='cuda' if torch.cuda.is_available() else 'cpu'; model.to(device).eval()
def generate(q,idx):
    context='\n'.join(f'{D.iloc[i].doc_id}: {D.iloc[i].text}' for i in idx)
    prompt=f'Responde solo con el número solicitado o SIN_EVIDENCIA. Pregunta: {q} Evidencia: {context}'
    inputs=tok(prompt,return_tensors='pt',truncation=True,max_length=512).to(device)
    with torch.no_grad(): out=model.generate(**inputs,max_new_tokens=20)
    return tok.decode(out[0],skip_special_tokens=True)
print(generate(V.question.iloc[0],dense(V.question.iloc[0],5)))
